<a href="https://colab.research.google.com/github/IngLuisGarcia17/EverPeak_clean_share/blob/main/EverPeak_clean_share.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EverPeak Retail Analysis PIPELINE

## Introducción

Hemos recibido el dataset EverPeak, el cual tenemos como objetivo crear PIPELINE de limpieza para dejar reutilizable y poder usarlo cuando se actualice el data o se vuelva a cargar.

## Cargar y Explorar

Antes de aplicar reglas de limpieza o automatizar pasos, necesitamos entender qué contiene realmente el dataset: qué columnas existen, qué representan y qué tan confiable es cada una.

In [14]:
# Importamos las librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [15]:
# Cargamos el archivo y visualizamos

df_clean = pd.read_csv('sample_data/everpeak_retail_raw.csv')
df_clean.head()

,order_id,order_date,customer_id,product_category,price,quantity,order_value,payment_method,city,state,customer_age
0,1,2024-02-02,2616,Sports,269,50,13385,credit_card,New York,NY,66.0
1,2,2024-10-10,1736,Grocery,66,0,660,debit_card,Los Angeles,CA,24.0
2,3,2024-08-27,2543,Sports,267,0,5073,credit_card,Chicago,IL,23.0
3,4,2024-06-09,2252,Toys,114,125,14290,credit_card,New York,NY,70.0
4,5,2024-06-07,1583,Fashion,729,16,11754,credit_card,Houston,TX,75.0


### Explorar la estructura de los datos

In [16]:
# Usamos info para explorar los datos
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5008 entries, 0 to 5007
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          5008 non-null   int64  
 1   order_date        5000 non-null   object 
 2   customer_id       5008 non-null   int64  
 3   product_category  5008 non-null   object 
 4   price             5008 non-null   int64  
 5   quantity          5008 non-null   int64  
 6   order_value       5008 non-null   int64  
 7   payment_method    5008 non-null   object 
 8   city              4908 non-null   object 
 9   state             4908 non-null   object 
 10  customer_age      4858 non-null   float64
dtypes: float64(1), int64(5), object(5)
memory usage: 430.5+ KB


Observamos **que order_id y customer_id** aparecen como int . Su función real es de IDs o claves, que identifican transacciones o clientes. Por ello no se analizan estadísticamente.

Luego vemos columnas como **customer_age ,  quantity y order_value.** lo que confirma que son variables numéricas y se pueden analizar.

También aparecen **product_category y city, entre otras,** como tipo object, pero en este caso significan columnas de categoría.

Por ultimo vemos columnas que contienen valores nulos.

In [17]:
# Cardinalidad de datos: inválidos / nan
df_clean.nunique().sort_values()

,0
payment_method,4
product_category,8
state,9
city,10
customer_age,64
quantity,249
order_date,381
price,1677
customer_id,1829
order_value,3889


In [18]:
# Porcentaje de nulos
df_clean.isna().mean().sort_values(ascending= False)

,0
customer_age,0.029952
city,0.019968
state,0.019968
order_date,0.001597
order_id,0.000000
customer_id,0.000000
product_category,0.000000
order_value,0.000000
quantity,0.000000
price,0.000000


* customer_age (2.99%): Es el campo con más ausencias. Aunque no es crítico para cálculos operativos, puede afectar análisis demográficos o segmentación de clientes.
* city (1.99%) y state (1.99%): Los datos de ubicación también presentan valores faltantes moderados. Esto podría impactar análisis geográficos o reportes por región.
* order_date (0.16%): La ausencia aquí es mínima, pero relevante, ya que podría afectar series temporales o análisis de ventas por fecha.

### Valores Inválidos

Aunque una valor no sea nulo debemos de ver que tan confiables los datos que si tenemos, por lo que efectuaremos codigo para validar los datos que tenemos en las columnas que presentaros mas alta cardinalidad de nulos.

In [19]:
df_clean['product_category'].value_counts()

,count
product_category,
Fashion,740
Electronics,736
Beauty,721
Toys,719
Sports,704
Grocery,684
Home,679
?,25


In [20]:
# Visualizamos la clasificacíon de "?"
df_clean[df_clean['product_category'].isin(['?'])]

,order_id,order_date,customer_id,product_category,price,quantity,order_value,payment_method,city,state,customer_age
212,213,2024-10-15,1119,?,356,0,2848,credit_card,Los Angeles,CA,30.0
568,569,2024-08-23,2279,?,2652,0,50388,paypal,Miami,FL,32.0
631,632,2024-04-29,1540,?,158,0,1106,credit_card,Denver,CO,54.0
831,832,2024-03-29,1171,?,185,0,370,credit_card,Denver,CO,49.0
1001,1002,2024-09-26,1948,?,176,0,1584,debit_card,San Francisco,CA,52.0
1195,1196,2024-08-19,1545,?,1848,0,25872,debit_card,Seattle,WA,49.0
1291,1292,2024-11-10,1377,?,84,0,84,credit_card,San Francisco,CA,44.0
1359,1360,2024-05-13,2845,?,3727,3,11165,credit_card,Chicago,IL,58.0
1363,1364,2024-03-14,2298,?,458,0,2748,credit_card,Seattle,WA,41.0
1473,1474,2024-03-16,2130,?,362,0,3620,paypal,Denver,CO,51.0


In [21]:
# Revisamos valores en quantity, menores o iguales a 0
df_clean['quantity'].le(0).sum()

np.int64(2971)

Tenemos mas de dos mil registros donde la cantidad del pedido es igual o menor a cero.  Este es un caso bastante grave, ya que representan aproximadamente la mitad de los registros.

In [22]:
df_clean['customer_age'].isin([-999]).sum()

np.int64(25)

Solo son 25 lo cual nos confirma cuántos casos deben imputarse por el valor correcto o reemplazarse por NaN más adelante en el pipeline.

## Definir funciones

Una vez que ya identificamos los principales errores podemos comenzar a definir lo que haran las funciones.

### Mini Funciones

In [23]:
# Funcion reemplazar sentinels, Funcion 1:
def reemplazar_sentinels_global(df,numeric_cols,text_cols):
    numeric_sentinels=[-999,999]
    for col in numeric_cols:
        df[col]=pd.to_numeric(df[col], errors = "coerce")
        df[col]=df[col].replace(numeric_sentinels, pd.NA)
    if 'quantity' in df.columns:
        df.loc[df['quantity'] <= 0, 'quantity'] = np.nan

    text_sentinels=['?']
    for col in text_cols:
        df[col]=df[col].replace(text_sentinels, 'uknown')
    return df

In [24]:
# Antes de imputar nulos, creamos la funcion para marcar con flags los valores que se sustituirán, Función 3:
def crear_flags(df, flags_cols):
    for col in flags_cols:
        nombre_flag = col + '_missing_flag'
        df[nombre_flag]= df[col].isna().astype(int)
    return df

In [25]:
# Funcion imputar nulos segun diagnostico, Función 3:

def imputar_segun_diagnostico(df, logic_fill_cols, median_fill_cols, fill_unknown_cols, date_drop_cols):
    # rellenar con Fórmula: quantity = value / price
    for col in logic_fill_cols:
        reemplazo = df['order_value'] // df['price']
        df['quantity'] = df['quantity'].fillna(reemplazo)

    # rellenar con la mediana en columnas numéricas
    for col in median_fill_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        med = df[col].median()
        df[col] = df[col].fillna(med)

    for col in (logic_fill_cols + median_fill_cols):
        if col in df.columns:
            df[col] = df[col].fillna(0).round().astype(int)

    # rellenar con texto "unknown" en columnas categóricas
    for col in fill_unknown_cols:
        df[col] = df[col].fillna("unknown")

    # eliminar registros con valores ausentes en columnas tipo fecha
    for col in date_drop_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        df = df.dropna(subset=[col]).reset_index(drop=True)
    return df

In [26]:
# Creamos la función general

def clean_data(df, numeric_cols, text_cols, flags_cols, logic_fill_cols, median_fill_cols,
               fill_unknown_cols, date_drop_cols):
    #funcion 1
    df = reemplazar_sentinels_global(df, numeric_cols, text_cols)

    #funcion 2
    df = crear_flags(df, flags_cols)

    #funcion 3
    df = imputar_segun_diagnostico(df, logic_fill_cols, median_fill_cols, fill_unknown_cols, date_drop_cols)

    return df

In [27]:
# Definimos la columnas correspondiendo a su tipo, función 1:
columnas_numericas=['customer_age','quantity']
columnas_texto=['product_category']

In [28]:
# Definimos las columnas función 2:
columnas_flags=['customer_age','city','state','quantity']

In [29]:
# Definimos columnas función 3:
cols_imputar_logica = ['quantity']
cols_imputar_mediana = ["customer_age"]
cols_imputar_unknown = ["city", "state"]
cols_imputar_fecha = ["order_date"]

In [30]:
# Aplicamos funcion general
df_share = clean_data(df_clean, columnas_numericas, columnas_texto,columnas_flags, cols_imputar_logica,
                      cols_imputar_mediana, cols_imputar_unknown, cols_imputar_fecha)



## Segmentación por grupos

En nuestro dataset, cada fila representa una orden individual: edad del cliente, valor del pedido, etc. Pero ninguna columna indica el tipo de cliente.

El equipo de estrategia e integración necesita saber:

* ¿Cuántos clientes son Senior VIP?
* ¿Cuáles aportan la mayoría de la ganancia?
* ¿Qué segmentos muestran más outliers?
* ¿Dónde están las oportunidades de retención?

Debemos crear una nueva característica:
un segmento de cliente basado en edad y gasto.

In [31]:
# Creamos la función que segmenta los clientes
def classify_segment(row):
  age = row['customer_age']
  spend = row['order_value']

  if pd.isna(age) or pd.isna(spend):
    return 'Error en Datos'

  if spend >= 10000:
    if age >= 55:
      return 'Senior VIP'
    else:
      return 'Junior VIP'

  elif spend >= 5000:
    if age >= 55:
      return 'Sr. Medium Value'
    else:
      return 'Jr. Medium Value'
  else:
      return 'Low Value'

In [32]:
df_share["customer_segment"] = df_share.apply(classify_segment, axis=1)
df_share[["customer_age", "order_value", "customer_segment"]].head()

,customer_age,order_value,customer_segment
0,66,13385,Senior VIP
1,24,660,Low Value
2,23,5073,Jr. Medium Value
3,70,14290,Senior VIP
4,75,11754,Senior VIP


In [33]:
# Guardamos el dataset nuevo
# df_share.to_csv('everpeak_clean_share.csv', index = False)